# SOHO matched-selection: locked test-only evaluation
This notebook never searches hyperparameters. It restores the original train-only `selection.json` evidence for CIFAR-100, CUB-200-2011 and ImageNet-R, verifies it against a committed preregistration manifest, creates an immutable authorization, then evaluates SOHO replay, official FLY, validation-tuned FLY and raw Ridge over six paired test replicates.

In [ ]:
# === Edit repository/path values only. ===
from pathlib import Path
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
SCRATCH_ROOT='/kaggle/temp' if Path('/kaggle').exists() else '/content'
PERSISTENT_ROOT='/kaggle/working' if Path('/kaggle/working').exists() else '/content'
WORK_DIR=f'{SCRATCH_ROOT}/SOHO-CL'
FEATURE_CACHE_ROOT=f'{SCRATCH_ROOT}/soho_matched_test_features'
SELECTION_ROOT=f'{SCRATCH_ROOT}/soho_matched_locked_selection'
OUTPUT_ROOT=f'{PERSISTENT_ROOT}/soho_matched_test_results'
SELECTION_ARTIFACT=''  # Optional explicit ZIP/directory; otherwise auto-detect under /kaggle/input or /content.
BATCH_SIZE=128
NUM_WORKERS=2
EXPECTED_PROTOCOL_SHA256='ab3e7fde1d375a96231ac26508926c21cbe74eee2474b1743b15589916694aeb'
EXPECTED_RUNNER_SHA256='4805a2192839314bb3710432e1e6aa4019b53d37cf798a0594957dc201e9ec97'
EXPECTED_BASE_RUNNER_SHA256='5d3ee38b93a10f4fdfa52aaeb1b901df97b0d40e5c806b67a2d609da094234fc'
EXPECTED_LOCKED_HPARAMS_SHA256='07b64b1fa9e57a1fbf08071c510b19bac06fcf7c6328944594d6e50a2b1258fa'

In [ ]:
# Fresh source checkout, dependencies, GPU check and immutable identity verification.
import hashlib,json,os,shutil,subprocess,sys,time
Path(SCRATCH_ROOT).mkdir(parents=True,exist_ok=True); os.chdir(SCRATCH_ROOT)
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib','seaborn'],check=True)
import torch
assert torch.cuda.is_available(),'Enable a GPU runtime.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
PROTOCOL='configs/soho_matched_selection_final.json'; RUNNER='tools/soho_matched_selection.py'; BASE_RUNNER='tools/soho_selfcontained.py'; LOCKED_HPARAMS='configs/soho_matched_selected_hyperparameters.json'
assert sha(PROTOCOL)==EXPECTED_PROTOCOL_SHA256
assert sha(RUNNER)==EXPECTED_RUNNER_SHA256
assert sha(BASE_RUNNER)==EXPECTED_BASE_RUNNER_SHA256
assert sha(LOCKED_HPARAMS)==EXPECTED_LOCKED_HPARAMS_SHA256
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('GPU:',torch.cuda.get_device_name(0)); print('commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()); print('LOCKED TEST SOURCE CHECK: PASS')

In [ ]:
# Restore the original selection evidence, then verify exact equality with the committed manifest.
search_roots=[Path('/kaggle/input'),Path('/content'),Path('/kaggle/working')]
explicit=Path(SELECTION_ARTIFACT) if SELECTION_ARTIFACT else None
candidates=[]
if explicit: candidates=[explicit]
else:
    for root in search_roots:
        if root.exists(): candidates += list(root.rglob('soho_matched_selection_train_only.zip')) + list(root.rglob('soho_matched_selection_resume.zip'))
assert candidates,f'Attach the train-only selection ZIP or set SELECTION_ARTIFACT. Searched: {search_roots}'
source=candidates[0]; staging=Path(SCRATCH_ROOT)/'selection_restore_staging'
if staging.exists(): shutil.rmtree(staging)
staging.mkdir(parents=True)
if source.is_file(): shutil.unpack_archive(str(source),str(staging))
elif source.is_dir(): shutil.copytree(source,staging,dirs_exist_ok=True)
else: raise FileNotFoundError(source)
roots=[]
for path in staging.rglob('cifar100/selection.json'):
    root=path.parent.parent
    if all((root/key/'selection.json').is_file() for key in ('cifar100','cub200','imagenetr')): roots.append(root)
assert len(roots)==1,f'Expected exactly one complete three-dataset selection root, found {roots}'
target=Path(SELECTION_ROOT)
if target.exists(): shutil.rmtree(target)
shutil.copytree(roots[0],target)
manifest=json.loads(Path(LOCKED_HPARAMS).read_text()); selected_rows=[]
for key in ('cifar100','cub200','imagenetr'):
    payload=json.loads((target/key/'selection.json').read_text()); expected=manifest['selected'][key]
    assert payload['status']=='SELECTION_COMPLETE'
    assert payload['uses_test_set'] is False and payload['held_out_test_authorized'] is False
    assert payload['protocol_sha256']==manifest['selection_protocol_sha256']
    assert payload['runner_sha256']==manifest['selection_runner_sha256']
    assert payload['base_runner_sha256']==manifest['base_runner_sha256']
    assert payload['selected_soho_config']==expected['soho_config']
    assert payload['selected_fly_config']==expected['fly_validation_tuned_config']
    assert float(payload['selected_raw_ridge_lambda'])==float(expected['raw_ridge_lambda'])
    selected_rows.append({'dataset':key,'SOHO':payload['selected_soho_config'],'FLY tuned':payload['selected_fly_config'],'raw lambda':payload['selected_raw_ridge_lambda'],'selection_sha256':sha(target/key/'selection.json')})
import pandas as pd
display(pd.DataFrame(selected_rows)); print('THREE-DATASET TRAIN-ONLY SELECTION EVIDENCE: PASS')

In [ ]:
# Download the verified checkpoint and resolve all processed datasets.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714 and sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATASET_ROOTS={'cifar100':kagglehub.dataset_download('zaphat206/cifar-100'),'cub200':kagglehub.dataset_download('zaphat206/cub-200-2011'),'imagenetr':kagglehub.dataset_download('zaphat206/imagenet-r')}
print(json.dumps(DATASET_ROOTS,indent=2))

In [ ]:
# Audit dataset identities before any test feature extraction.
CUB_AUDIT=f'{SCRATCH_ROOT}/cub_soho_matched_test_audit.json'; IMAGENETR_AUDIT=f'{SCRATCH_ROOT}/imagenetr_soho_matched_test_audit.json'
assert subprocess.run([sys.executable,'-u','tools/cub_dataset_audit.py','--root',DATASET_ROOTS['cub200'],'--output',CUB_AUDIT,'--expected-identity-sha256','e374af9b576cb6b3503198ef3ea30fd0aa9d2e18c230ff8064e21d4f644af2ca']).returncode==0
assert subprocess.run([sys.executable,'-u','tools/imagenetr_dataset_audit.py','--root',DATASET_ROOTS['imagenetr'],'--output',IMAGENETR_AUDIT,'--expected-identity-sha256','3f3d963b2b0c245ceabc0166c8b1c64d624c2ea31df07ee6ffdbf4cab5f7739d','--diagnose-cross-split-duplicates','--workers','4']).returncode==2
overlap=json.loads(Path(IMAGENETR_AUDIT).read_text()); assert overlap['cross_split_duplicate_content_count']==19 and overlap['cross_split_conflicting_label_duplicate_count']==18
print('DATASET IDENTITY AUDIT: PASS; ImageNet-R remains labeled legacy processed split.')

In [ ]:
# Extract frozen TRAIN features only. Test remains unavailable.
protocol=json.loads(Path(PROTOCOL).read_text()); Path(FEATURE_CACHE_ROOT).mkdir(parents=True,exist_ok=True)
for key in ('cifar100','cub200','imagenetr'):
    cfg=protocol['datasets'][key]; cache=Path(FEATURE_CACHE_ROOT)/key
    if not (cache/'train.pt').is_file():
        command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256',protocol['backbone']['checkpoint_sha256'],'--feature-cache-dir',str(cache),'--output-dir',f'{SCRATCH_ROOT}/unused_test_{key}','--dataset',cfg['dataset'],'--model-name',protocol['backbone']['model_name'],'--data-augmentation','vit','--seed','2025','--num-classes',str(cfg['num_classes']),'--num-tasks',str(cfg['num_tasks']),'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
        print(f'TRAIN EXTRACT START {key}',flush=True); subprocess.run(command,check=True)
    assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists(); print(f'TRAIN CACHE READY {key}; test.pt absent')
print('ALL TRAIN CACHES READY; TEST STILL HIDDEN')

In [ ]:
# Correctness gate and immutable authorization. No accuracy is inspected here.
command=[sys.executable,'-B','-m','pytest','-q','tests/test_soho_matched_selection.py','tests/test_soho_selfcontained.py','tests/test_cached_replay_baselines.py']
completed=subprocess.run(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True); print(completed.stdout,flush=True); assert completed.returncode==0
dirty=subprocess.check_output(['git','status','--porcelain'],text=True).strip(); assert not dirty,f'Repository changed before authorization:\n{dirty}'
Path(OUTPUT_ROOT).mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'-u',RUNNER,'lock','--protocol',PROTOCOL,'--selection-root',SELECTION_ROOT,'--output-root',OUTPUT_ROOT,'--require-clean-git'],check=True)
AUTHORIZATION=str(Path(OUTPUT_ROOT)/'authorization.json'); print(json.dumps(json.loads(Path(AUTHORIZATION).read_text()),indent=2)); print('AUTHORIZED TEST BOUNDARY: LOCKED')

## Test begins below
The selection files, code hashes, manifest and Git commit are now immutable. Do not change any hyperparameter after running the next cell.

In [ ]:
# Materialize TEST features only after authorization.
for key in ('cifar100','cub200','imagenetr'):
    command=[sys.executable,'-u',RUNNER,'extract-test','--protocol',PROTOCOL,'--dataset-key',key,'--selection-root',SELECTION_ROOT,'--authorization',AUTHORIZATION,'--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print(f'TEST EXTRACTION START {key}',flush=True); subprocess.run(command,check=True)
print('ALL AUTHORIZED TEST FEATURE CACHES READY')

In [ ]:
# Final helper: refit from empty state on full train and run six paired replicates x four methods.
audit_paths={'cifar100':None,'cub200':CUB_AUDIT,'imagenetr':IMAGENETR_AUDIT}
def run_final(key):
    command=[sys.executable,'-u',RUNNER,'evaluate','--protocol',PROTOCOL,'--dataset-key',key,'--selection-root',SELECTION_ROOT,'--authorization',AUTHORIZATION,'--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--output-root',OUTPUT_ROOT,'--device','cuda']
    if audit_paths[key]: command += ['--dataset-audit',audit_paths[key]]
    print(f'FINAL START {key}: 6 paired replicates x 4 methods',flush=True); subprocess.run(command,check=True)
    payload=json.loads(Path(OUTPUT_ROOT,key,'final_results.json').read_text()); rows=[]
    for replicate in payload['seed_results']:
        for method,result in replicate['methods'].items(): rows.append({'replicate':replicate['replicate_index'],'method':method,'status':result['status'],'final_accuracy':result.get('final_accuracy'),'AIA':result.get('average_incremental_accuracy'),'forgetting':result.get('forgetting'),'state_MiB':None if result.get('persistent_state_bytes') is None else result['persistent_state_bytes']/2**20,'exemplar_free':result.get('state_audit',{}).get('exemplar_free')})
    display(pd.DataFrame(rows)); print('STATUS:',payload['status'])

In [ ]:
# CIFAR-100 test.
run_final('cifar100')

In [ ]:
# CUB-200-2011 test.
run_final('cub200')

In [ ]:
# ImageNet-R legacy processed-split test.
run_final('imagenetr')

In [ ]:
# Aggregate means, sample standard deviations, CIs, paired differences and plots.
subprocess.run([sys.executable,'-u',RUNNER,'summarize','--protocol',PROTOCOL,'--output-root',OUTPUT_ROOT],check=True)
import matplotlib.pyplot as plt,seaborn as sns
metrics=pd.read_csv(Path(OUTPUT_ROOT)/'metrics_summary.csv'); display(metrics.sort_values(['dataset','method']))
summary=json.loads(Path(OUTPUT_ROOT,'final_summary.json').read_text()); print(json.dumps(summary['paired_aia_differences'],indent=2))
curves=pd.read_csv(Path(OUTPUT_ROOT)/'task_curves.csv'); names={'soho_replay_fidelity':'SOHO replay','flycl_fidelity':'FLY official','flycl_validation_tuned':'FLY tuned','raw_ridge':'Raw Ridge'}
fig,axes=plt.subplots(1,3,figsize=(21,5.5),sharey=True)
for ax,key in zip(axes,('cifar100','cub200','imagenetr')):
    view=curves[curves.dataset==key]
    for method,label in names.items():
        mean=view[view.method==method].groupby('task').average_seen_accuracy.mean(); ax.plot(mean.index,mean.values,label=label,marker='o',ms=3)
    ax.set_title(key); ax.set_xlabel('Task'); ax.set_ylabel('Average seen-class accuracy (%)')
axes[0].legend(fontsize=9); fig.tight_layout(); fig.savefig(Path(OUTPUT_ROOT)/'locked_test_accuracy_curves.png',dpi=220,bbox_inches='tight'); plt.show()

In [ ]:
# Export compact evidence; caches and replay tensors are excluded. Kaggle retains files under /kaggle/working.
shutil.copy2(PROTOCOL,Path(OUTPUT_ROOT)/'locked_protocol.json'); shutil.copy2(RUNNER,Path(OUTPUT_ROOT)/'locked_runner.py'); shutil.copy2(BASE_RUNNER,Path(OUTPUT_ROOT)/'locked_base_runner.py'); shutil.copy2(LOCKED_HPARAMS,Path(OUTPUT_ROOT)/'locked_selected_hyperparameters.json')
selection_copy=Path(OUTPUT_ROOT)/'train_only_selection'; selection_copy.mkdir(exist_ok=True)
for key in ('cifar100','cub200','imagenetr'): shutil.copy2(Path(SELECTION_ROOT,key,'selection.json'),selection_copy/f'{key}_selection.json')
archive=shutil.make_archive(str(Path(PERSISTENT_ROOT)/'soho_matched_locked_test_results'),'zip',root_dir=OUTPUT_ROOT)
print('artifact:',archive,'size=',Path(archive).stat().st_size,'SHA-256=',sha(archive))
if not Path('/kaggle/working').exists():
    from google.colab import files
    files.download(archive)